In [12]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
from sklearn.preprocessing import LabelEncoder

dagshub.init(repo_owner='mkakh22', repo_name='house-prices-mlflow', mlflow=True)

print("Setup complete!")

Initialized MLflow to track repo "mkakh22/house-prices-mlflow"

Repository mkakh22/house-prices-mlflow initialized!

Setup complete!


In [13]:
test = pd.read_csv('test.csv')
train = pd.read_csv('train.csv')
print(f"Test shape: {test.shape}")

Test shape: (1459, 80)


In [18]:
from sklearn.ensemble import RandomForestRegressor

train_clean, test_clean = preprocess_data(train, test)

X_all_train = train_clean.drop(columns=['SalePrice'])
y_train = np.log(train_clean['SalePrice'])
X_all_test = test_clean

rf_selector = RandomForestRegressor(n_estimators=100, random_state=42)
rf_selector.fit(X_all_train, y_train)

importances = pd.Series(rf_selector.feature_importances_, index=X_all_train.columns)
features_rf = importances.nlargest(25).index.tolist()

X_train = X_all_train[features_rf]
X_test  = X_all_test[features_rf]

print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"Features: {features_rf}")
print("Data preprocessed!")

Train shape: (1460, 25)
Test shape:  (1459, 25)
Features: ['OverallQual', 'TotalSF', 'CentralAir', 'GrLivArea', 'GarageCars', 'GarageArea', 'LotArea', 'BsmtFinSF1', 'OverallCond', 'RemodAge', 'HouseAge', 'BsmtUnfSF', '1stFlrSF', 'GarageType', 'YearRemodAdd', 'YearBuilt', 'Neighborhood', 'MSZoning', 'LotFrontage', 'TotalBsmtSF', '2ndFlrSF', 'Fireplaces', 'BsmtQual', 'GarageYrBlt', 'TotalBath']
Data preprocessed!


In [19]:
model_name = "house-prices-best-model"
model_version = "2"

model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.sklearn.load_model(model_uri)

print(f" Model loaded from Registry!")
print(f"   URI: {model_uri}")
print(f"   Model type: {type(model).__name__}")

 Model loaded from Registry!
   URI: models:/house-prices-best-model/2
   Model type: XGBRegressor


In [20]:
log_predictions = model.predict(X_test)

predictions = np.expm1(log_predictions)

print(f"Predictions shape: {predictions.shape}")
print(f"Sample predictions: {predictions[:5]}")
print(f"Min: ${predictions.min():,.0f}")
print(f"Max: ${predictions.max():,.0f}")
print(f"Mean: ${predictions.mean():,.0f}")

Predictions shape: (1459,)
Sample predictions: [127235.914 151346.5   182689.9   185362.06  188837.66 ]
Min: $40,016
Max: $526,903
Mean: $177,226


In [21]:
submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': predictions
})

submission.to_csv('submission.csv', index=False)

print(" Submission saved!")
print(f"Shape: {submission.shape}")
print(submission.head(10))

 Submission saved!
Shape: (1459, 2)
     Id      SalePrice
0  1461  127235.914062
1  1462  151346.500000
2  1463  182689.906250
3  1464  185362.062500
4  1465  188837.656250
5  1466  175724.984375
6  1467  177484.828125
7  1468  169207.515625
8  1469  180876.718750
9  1470  118729.476562
